# 第3回講義 宿題

## 課題

今回のLessonで学んだことを元に，MNISTのファッション版 (Fashion MNIST，クラス数10) を多層パーセプトロンによって分類してみましょう．

Fashion MNISTの詳細については以下のリンクを参考にしてください．

Fashion MNIST: https://github.com/zalandoresearch/fashion-mnist

### 目標値

Accuracy 85%

### ルール

- 訓練データは`x_train`， `t_train`，テストデータは`x_test`で与えられます．
- 予測ラベルは one_hot表現ではなく0~9のクラスラベル で表してください．
- **下のセルで指定されている`x_train`，`t_train`以外の学習データは使わないでください．**
- **多層パーセプトロンのアルゴリズム部分は第3回の演習を参考に，NumPyのみで実装してください．** (sklearnやtensorflowなどは使用しないでください)．
    - データの前処理部分でsklearnの関数を使う (例えば `sklearn.model_selection.train_test_split`) のは問題ありません．

### 提出方法
- 2つのファイルを提出していただきます．
    1. テストデータ (`x_test`) に対する予測ラベルを`submission_pred.csv`として保存し，**Omnicampusの宿題タブから「第3回 ニューラルネットワーク基礎」を選択して**提出してください．
    2. それに対応するpythonのコードを`submission_code.py`として保存し，**Omnicampusの宿題タブから「第3回 ニューラルネットワーク基礎 (code)」を選択して**提出してください．pythonファイル自体の提出ではなく，「提出内容」の部分にコードをコピー&ペーストしてください．
      
- なお，採点は1で行い，2はコードの確認用として利用します（成績優秀者はコード内容を公開させていただくかもしれません）．コードの内容を変更した場合は，**1と2の両方を提出し直してください**．

### 評価方法
- 予測ラベルの`t_test`に対する精度 (Accuracy) で評価します．
- 即時採点しLeader Boardを更新します（採点スケジュールは別アナウンス）．
- 締切時の点数を最終的な評価とします．

### ドライブのマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### データの読み込み（このセルは修正しないでください）

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import inspect


#学習データ
x_train = np.load('drive/MyDrive/DLBasic/HW/HW3/data/x_train.npy')
t_train = np.load('drive/MyDrive/DLBasic/HW/HW3/data/y_train.npy')

#テストデータ
x_test = np.load('drive/MyDrive/DLBasic/HW/HW3/data/x_test.npy')

# データの前処理（正規化， one-hot encoding)
x_train, x_test = x_train / 255., x_test / 255.
x_train, x_test = x_train.reshape(x_train.shape[0], -1), x_test.reshape(x_test.shape[0], -1)
t_train = np.eye(N=10)[t_train.astype("int32").flatten()]

FileNotFoundError: [Errno 2] No such file or directory: 'drive/MyDrive/DLBasic/HW/HW3/data/x_train.npy'

### 多層パーセプトロンの実装

In [ ]:
# データの分割
x_train, x_val, t_train, t_val =\
    train_test_split(x_train, t_train, test_size=10000)

In [ ]:
def np_log(x):
    return np.log(np.clip(x, 1e-10, 1e+10))


def create_batch(data, batch_size):
    """
    :param data: np.ndarray，入力データ
    :param batch_size: int，バッチサイズ
    """
    num_batches, mod = divmod(data.shape[0], batch_size)
    batched_data = np.split(data[: batch_size * num_batches], num_batches)
    if mod:
        batched_data.append(data[batch_size * num_batches:])

    return batched_data

In [ ]:
class AdamOptimizer:
    def __init__(self, shape, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = np.zeros(shape)
        self.v = np.zeros(shape)
        self.t = 0

    def update(self, param, grad, lr):
        self.lr = lr
        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1 - self.beta2) * (grad ** 2)
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)
        return param - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

In [ ]:
# シード値を変えることで何が起きるかも確かめてみてください．
rng = np.random.RandomState(1234)
random_state = 42


# 発展: 今回の講義で扱っていない活性化関数について調べ，実装してみましょう
def relu(x):
    return np.maximum(x, 0)


def deriv_relu(x):
    return (x > 0).astype(x.dtype)

def gelu(x):
    return 0.5 * x * (1 + torch.tanh(math.sqrt(math.pi / 2) * (x + 0.044715 * x ** 3)))

def deriv_gelu(x):
    tanh_term = np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3))
    sech2 = 1 - tanh_term**2
    return 0.5 * (1 + tanh_term) + 0.5 * x * sech2 * np.sqrt(2/np.pi) * (1 + 3 * 0.044715 * x**2)

def elu(x, alpha=1.0):
    return np.where(x > 0.0, x, alpha * (np.exp(x) - 1))

def deriv_elu(x, alpha=1.0):
    return np.where(x > 0.0, 1.0, elu(x, alpha) + alpha)

def LeakyRelu(x):
    return np.where(x > 0, x, 0.01 * x)

def deriv_LeakyRelu(x):
    return np.where(x > 0, 1, 0.01)

def softmax(x):
    x -= x.max(axis=1, keepdims=True)
    x_exp = np.exp(x)
    return x_exp / np.sum(x_exp, axis=1, keepdims=True)


def deriv_softmax(x):
    return np.ones_like(x)


def crossentropy_loss(t, y):
    return (-t * np_log(y)).sum(axis=1).mean()


class Dense:
    def __init__(self, in_dim, out_dim, function, deriv_function):
      # self.W = rng.randn(in_dim, out_dim) / np.sqrt(in_dim) * np.sqrt(2) # He初期化
      self.W = rng.uniform(low=-0.08, high=0.08, size=(in_dim, out_dim)).astype("float64")
      self.b = np.zeros(out_dim).astype("float64")
      self.function = function
      self.deriv_function = deriv_function

      self.x = None
      self.u = None

      self.dW = None
      self.db = None

      self.params_idx = np.cumsum([self.W.size, self.b.size])

      self.optimizer_W = AdamOptimizer(self.W.shape)
      self.optimizer_b = AdamOptimizer(self.b.shape)

    def __call__(self, x):
        """
        順伝播処理を行う。
        Args:
            x: (batch_size, in_dim_{j})
        Returns:
            h: (batch_size, out_dim_{j})
        """
        self.x = x
        self.u = np.matmul(self.x, self.W) + self.b
        h = self.function(self.u)

        return h

    def b_prop(self, delta, W):
        """
        誤差逆伝播を行う。
        Args:
            delta (=delta_{j+1}): (batch_size, out_dim_{j+1})
            W (W_{j+1}): (out_dim_{j}, out_dim_{j+1})
        Retuens:
            self.delta (=delta_{j}): (batch_size, out_dim_{j})
        """
        self.delta = self.deriv_function(self.u) * np.matmul(delta, W.T)

        return self.delta
    def compute_grad(self):
        """
        勾配(dW, db)を計算する。
        """
        batch_size = self.delta.shape[0]

        self.dW = np.matmul(self.x.T, self.delta) / batch_size
        self.db = np.matmul(np.ones(batch_size), self.delta) / batch_size

    def get_params(self):
        """
        Returns:
            self.Wとself.bを結合して、平坦化したもの
        """
        return np.concatenate([self.W.ravel(), self.b], axis=0)

    def set_params(self, params):
        """
        paramsからWとbを取得。
        Args:
            params: List[np.ndarray, np.ndarray]
                一つ目の要素がW(in_dim, out_dim)、二つ目の要素がb(out_dim,)
        """
        _W, _b = np.split(params, self.params_idx)[:-1]
        self.W = _W.reshape(self.W.shape)
        self.b = _b
    def get_grads(self):
        """
        Returns:
            self.dWとself.dbを結合して、平坦化したもの
        """
        return np.concatenate([self.dW.ravel(), self.db], axis=0)


class Model:
    def __init__(self, hidden_dims, activation_functions, deriv_functions):
        """
        Args:
            hidden_dims(List[int]): 各層のノード数を格納したリスト
            activation_functions(List): 各層で用いる活性化関数を格納したリスト
            deriv_functions(List): 各層で用いる活性化関数の導関数を格納したリスト
        """
        # 各層をリストに格納
        self.layers = []
        # 出力層以外
        for i in range(len(hidden_dims)-2):
            self.layers.append(Dense(hidden_dims[i], hidden_dims[i+1], activation_functions[i], deriv_functions[i]))
        # 出力層
        self.layers.append(Dense(hidden_dims[-2], hidden_dims[-1], activation_functions[-1], deriv_functions[-1]))
    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        """
        順伝播を行う。
        """
        for layer in self.layers:
            x = layer(x)
        return x
    def backward(self, delta):
        """
        誤差逆伝播、勾配計算を行う。
        """
        for i, layer in enumerate(self.layers[::-1]):
            if i == 0: # 出力層
                layer.delta = delta
                layer.compute_grad()
            else: # 出力層以外
                delta = layer.b_prop(delta, W) # 逆伝播
                layer.compute_grad() # 勾配計算
            W = layer.W

    def update(self, eps = 0.01):
        """
        パラメータの更新を行う
        """
        for layer in self.layers:
            layer.W = layer.optimizer_W.update(layer.W, layer.dW, eps)
            layer.b = layer.optimizer_b.update(layer.b, layer.db, eps)

            # layer.W -= eps * layer.dW
            # layer.b -= eps * layer.db


lr = 0.01
n_epochs = 10
batch_size = 256

mlp = Model(
            hidden_dims=[784, 392, 40, 10],
            activation_functions=[gelu, gelu, softmax],
            deriv_functions=[deriv_gelu, deriv_gelu, deriv_softmax]
      )

### モデルの学習

In [ ]:
"""### モデルの学習"""

def train_model(mlp, x_train, t_train, x_val, t_val, n_epochs=10):
    global lr

    for epoch in range(n_epochs):
        losses_train = []
        losses_valid = []
        train_num = 0
        train_true_num = 0
        valid_num = 0
        valid_true_num = 0

        x_train, t_train = shuffle(x_train, t_train, random_state=random_state)
        x_train_batches, t_train_batches = create_batch(x_train, batch_size), create_batch(t_train, batch_size)

        x_val, t_val = shuffle(x_val, t_val, random_state=random_state)
        x_val_batches, t_val_batches = create_batch(x_val, batch_size), create_batch(t_val, batch_size)

        # モデルの訓練
        for x, t in zip(x_train_batches, t_train_batches):
            # 順伝播
            y = mlp(x)

            # 損失の計算
            loss = crossentropy_loss(t, y)
            losses_train.append(loss.tolist())

            # 逆伝播
            delta = y - t
            mlp.backward(delta)

            # パラメータの更新
            mlp.update(lr)

            # 精度を計算
            acc = accuracy_score(t.argmax(axis=1), y.argmax(axis=1), normalize=False)
            train_num += x.shape[0]
            train_true_num += acc

        # lrの更新
        lr *= 0.8
        # if epoch == 3:
        #     lr = 0.008
        # elif epoch == 8:
        #     lr = 0.006

        # モデルの評価
        for x, t in zip(x_val_batches, t_val_batches):
            # 順伝播
            y = mlp(x)

            # 損失の計算
            loss = crossentropy_loss(t, y)
            losses_valid.append(loss.tolist())

            acc = accuracy_score(t.argmax(axis=1), y.argmax(axis=1), normalize=False)
            valid_num += x.shape[0]
            valid_true_num += acc

        print('EPOCH: {}, Train [Loss: {:.3f}, Accuracy: {:.3f}], Valid [Loss: {:.3f}, Accuracy: {:.3f}]'.format(
            epoch+1,
            np.mean(losses_train),
            train_true_num/train_num,
            np.mean(losses_valid),
            valid_true_num/valid_num
        ))


train_model(mlp, x_train, t_train, x_val, t_val, n_epochs)

In [ ]:
t_pred = []
for x in x_test:
    # 順伝播
    x = x[np.newaxis, :]
    y = mlp(x)

    # モデルの出力を予測値のスカラーに変換
    pred = y.argmax(1).tolist()

    t_pred.extend(pred)

submission = pd.Series(t_pred, name='label')
submission.to_csv('drive/MyDrive/DLBasic/HW/HW3/submission_pred.csv', header=True, index_label='id')